# 🚀 Task 2: Trainable Machine Learning Classifier on ArcFace Embeddings

### 📌 Research Overview & Technical Objectives
This notebook presents the **Trainable Machine Learning Expansion** of the face recognition pipeline. Instead of relying solely on static cosine template averaging, we extract individual 512D ArcFace embeddings for all gallery samples and train supervised scikit-learn classifiers (**Linear SVM, RBF SVM, Logistic Regression, k-NN**).

---

### 🔑 Key Research Concepts & Terminology
1. **Supervised Feature Space Learning**:
   - Maps 512D ArcFace embeddings $\mathbf{x} \in \mathbb{R}^{512}$ to enrolled identity class labels $y \in \{1, \dots, K\}$ using discriminative decision boundaries:
     $$P(y = k \mid \mathbf{x}) = \text{Softmax}(f_k(\mathbf{x}))$$
2. **Multi-Model Benchmark**:
   - **Linear SVM**: Hyperplane separation in high-dimensional embedding space.
   - **RBF Kernel SVM**: Non-linear boundary learning for complex distribution support.
   - **Logistic Regression**: Calibrated probabilistic log-odds regression.
   - **k-Nearest Neighbors (k-NN)**: Distance-weighted local neighborhood voting.
3. **Open-Set Probability Calibration**:
   - Supervised classifiers output probabilities $P(\text{Identity} \mid \mathbf{x})$. To handle unknown out-of-distribution faces, we calibrate a confidence rejection threshold $\tau_{\text{cls}}$:
     $$\text{Decision} = \begin{cases} \hat{y} & \text{if } \max_k P(y=k \mid \mathbf{x}) \ge \tau_{\text{cls}} \\ \text{"NOT RECOGNIZED"} & \text{otherwise} \end{cases}$$
4. **Model Artifact Serialization**:
   - Model parameters, feature scaling parameters, and label mappings are persisted via `joblib` (`best_classifier.joblib`, `scaler.joblib`, `label_encoder.joblib`, `threshold.json`).


In [ ]:
# 1. Environment Setup & Imports
import sys
import os
import subprocess

try:
    import insightface
    import cv2
    import sklearn
    import joblib
except ImportError:
    print("Installing required dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 
                            "insightface", "onnxruntime", "opencv-python", 
                            "scikit-learn", "matplotlib", "seaborn", "pyyaml", "pandas", "tqdm", "joblib", "requests"])
    import insightface
    import cv2
    import sklearn
    import joblib

import json
import time
import shutil
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report, 
    confusion_matrix, roc_curve, roc_auc_score
)

# Plotting config
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Trainable Pipeline Environment Ready.")


## 📦 Section 1: Per-Sample Gallery Embedding Dataset Extraction

We extract individual 512D ArcFace embeddings for every image in `gallery/` to create a dataset:
- Feature Matrix $\mathbf{X} \in \mathbb{R}^{N \times 512}$ (where $N = 150$ samples across 10 identities).
- Target Label Vector $\mathbf{y} \in \mathbb{R}^N$.
- We perform a **Stratified 80/20 Train/Test Split** to ensure balanced class distribution.


In [ ]:
# 2. Extract Per-Sample Embedding Dataset
ROOT_DIR = Path.cwd()
GALLERY_DIR = ROOT_DIR / "gallery"
TRAINING_DIR = ROOT_DIR / "training"
MODELS_DIR = ROOT_DIR / "models"
REPORTS_DIR = ROOT_DIR / "reports"
TEST_IMAGES_DIR = ROOT_DIR / "test_images"
KNOWN_TEST_DIR = TEST_IMAGES_DIR / "known"
UNKNOWN_TEST_DIR = TEST_IMAGES_DIR / "unknown"

for d in [TRAINING_DIR, MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def l2_normalize(vec: np.ndarray) -> np.ndarray:
    arr = np.asarray(vec, dtype=np.float32).reshape(-1)
    norm = np.linalg.norm(arr)
    return arr / norm if norm > 0 else arr

class FaceDetector:
    def __init__(self):
        from insightface.app import FaceAnalysis
        model_root = ROOT_DIR / ".insightface"
        model_root.mkdir(parents=True, exist_ok=True)
        self.app = FaceAnalysis(name="buffalo_l", root=str(model_root), providers=["CPUExecutionProvider"])
        self.app.prepare(ctx_id=0, det_size=(640, 640))
        
    def detect(self, img):
        faces = self.app.get(img)
        res = []
        for f in faces:
            emb = getattr(f, "normed_embedding", getattr(f, "embedding", None))
            if emb is not None: emb = l2_normalize(emb)
            res.append({"bbox": np.asarray(f.bbox, dtype=np.float32), "det_score": float(getattr(f, "det_score", 0.0)), "embedding": emb})
        return res

detector = FaceDetector()

def create_embedding_dataset():
    embeddings, labels = [], []
    id_dirs = sorted([d for d in GALLERY_DIR.iterdir() if d.is_dir()])
    
    print(f"Extracting embeddings across {len(id_dirs)} gallery folders...")
    for d in id_dirs:
        img_paths = sorted([p for p in d.iterdir() if p.suffix.lower() in {".jpg", ".png", ".jpeg"}])
        for p in img_paths:
            img = cv2.imread(str(p))
            if img is None: continue
            faces = detector.detect(img)
            if faces and faces[0]["embedding"] is not None:
                embeddings.append(faces[0]["embedding"])
                labels.append(d.name)
                
    X = np.stack(embeddings, axis=0).astype(np.float32)
    y = np.asarray(labels, dtype=str)
    
    np.save(TRAINING_DIR / "embeddings.npy", X)
    np.save(TRAINING_DIR / "labels.npy", y)
    print(f"Dataset generated! Matrix X shape: {X.shape}, Label vector y shape: {y.shape}")
    return X, y

X, y = create_embedding_dataset()


## 🏋️ Section 2: Multi-Model Supervised Training & Benchmark Evaluation

We split the dataset into **80% Training Set** and **20% Validation Set** (stratified).
We fit a `StandardScaler` and `LabelEncoder`, then train and evaluate four ML algorithms:
1. **Linear SVM** (`kernel="linear", probability=True`)
2. **RBF SVM** (`kernel="rbf", probability=True`)
3. **Logistic Regression** (`max_iter=3000`)
4. **k-Nearest Neighbors** (`n_neighbors=5, weights="distance"`)

We compare Accuracy, Precision, Recall, F1-Score, OVR ROC AUC, and select the top model.


In [ ]:
# 3. Model Training, Comparison & Selection
def train_and_evaluate_classifiers(X, y):
    encoder = LabelEncoder()
    y_enc = encoder.fit_transform(y)
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y_enc, test_size=0.20, random_state=42, stratify=y_enc
    )
    
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    models = {
        "Linear SVM": SVC(kernel="linear", probability=True, class_weight="balanced", random_state=42),
        "RBF SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, weights="distance")
    }
    
    results = []
    trained_objects = {}
    
    for name, clf in models.items():
        clf.fit(X_train_scaled, y_train)
        y_pred = clf.predict(X_val_scaled)
        y_proba = clf.predict_proba(X_val_scaled)
        
        acc = accuracy_score(y_val, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred, average="weighted", zero_division=0)
        
        results.append({
            "Classifier": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1
        })
        trained_objects[name] = clf
        
    res_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
    
    # PLOTTING COMPARISON
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Barplot comparison
    sns.barplot(data=res_df, x="Classifier", y="Accuracy", ax=axes[0], palette="viridis")
    axes[0].set_ylim(0, 1.05)
    axes[0].set_title("Classifier Validation Accuracy Comparison")
    for p in axes[0].patches:
        axes[0].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 5), textcoords='offset points')
                         
    # Top Model Confusion Matrix
    best_name = res_df.iloc[0]["Classifier"]
    best_clf = trained_objects[best_name]
    best_pred = best_clf.predict(X_val_scaled)
    cm = confusion_matrix(y_val, best_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                xticklabels=encoder.classes_, yticklabels=encoder.classes_, ax=axes[1])
    axes[1].set_title(f"Best Classifier ({best_name}) Confusion Matrix")
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Winner Model: {best_name} (Validation Accuracy: {res_df.iloc[0]['Accuracy']:.4f})")
    return best_clf, scaler, encoder, best_name, res_df

best_clf, scaler, encoder, best_name, res_df = train_and_evaluate_classifiers(X, y)
display(res_df)


## 🎛️ Section 3: Open-Set Probability Threshold Calibration

Since supervised classifiers output posterior probabilities $P(y = k \mid \mathbf{x})$, we must calibrate a **confidence threshold $\tau_{\text{cls}}$** to reject unknown faces.

We test thresholds $\tau_{\text{cls}} \in [0.50, 0.90]$ on separate Known and Unknown test sets to find the optimal decision cutoff.


In [ ]:
# 4. Threshold Calibration & Open-Set Evaluation
def calibrate_classifier_threshold(classifier, scaler, encoder):
    known_files = sorted(list(KNOWN_TEST_DIR.rglob("*.jpg")))
    unknown_files = sorted(list(UNKNOWN_TEST_DIR.rglob("*.jpg")))
    
    records = []
    
    for p in known_files:
        img = cv2.imread(str(p))
        if img is None: continue
        faces = detector.detect(img)
        if not faces: continue
        best_face = max(faces, key=lambda x: (x["bbox"][2]-x["bbox"][0])*(x["bbox"][3]-x["bbox"][1]))
        
        scaled_emb = scaler.transform([l2_normalize(best_face["embedding"])])
        probas = classifier.predict_proba(scaled_emb)[0]
        idx = int(np.argmax(probas))
        conf = float(probas[idx])
        pred_id = encoder.inverse_transform([idx])[0]
        
        records.append({"is_known": True, "actual": p.parent.name, "predicted": pred_id, "confidence": conf})
        
    for p in unknown_files:
        img = cv2.imread(str(p))
        if img is None: continue
        faces = detector.detect(img)
        if not faces: continue
        best_face = max(faces, key=lambda x: (x["bbox"][2]-x["bbox"][0])*(x["bbox"][3]-x["bbox"][1]))
        
        scaled_emb = scaler.transform([l2_normalize(best_face["embedding"])])
        probas = classifier.predict_proba(scaled_emb)[0]
        idx = int(np.argmax(probas))
        conf = float(probas[idx])
        pred_id = encoder.inverse_transform([idx])[0]
        
        records.append({"is_known": False, "actual": "UNKNOWN", "predicted": pred_id, "confidence": conf})
        
    calib_df = pd.DataFrame(records)
    
    thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]
    calib_rows = []
    
    for t in thresholds:
        accepted_correct_known = sum(r["is_known"] and r["predicted"] == r["actual"] and r["confidence"] >= t for _, r in calib_df.iterrows())
        known_total = sum(calib_df["is_known"])
        unknown_total = sum(~calib_df["is_known"])
        
        false_accepts = sum((not r["is_known"]) and r["confidence"] >= t for _, r in calib_df.iterrows())
        false_rejects = sum(r["is_known"] and (r["confidence"] < t or r["predicted"] != r["actual"]) for _, r in calib_df.iterrows())
        
        acc = accepted_correct_known / max(1, known_total)
        far = false_accepts / max(1, unknown_total)
        frr = false_rejects / max(1, known_total)
        f1 = 2 * (1 - far) * (1 - frr) / max(1e-5, (1 - far) + (1 - frr))
        
        calib_rows.append({"threshold": t, "known_acc": acc, "FAR": far, "FRR": frr, "score": f1})
        
    res_calib = pd.DataFrame(calib_rows)
    best_row = res_calib.loc[res_calib["score"].idxmax()]
    best_thresh = float(best_row["threshold"])
    
    # PLOTTING
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(res_calib["threshold"], res_calib["known_acc"], marker="o", label="Known Accuracy", color="green")
    ax.plot(res_calib["threshold"], res_calib["FAR"], marker="s", label="False Accept Rate (FAR)", color="red")
    ax.plot(res_calib["threshold"], res_calib["FRR"], marker="^", label="False Reject Rate (FRR)", color="orange")
    ax.axvline(best_thresh, color="black", linestyle="--", label=f"Optimal Threshold ({best_thresh:.2f})")
    ax.set_title("Probability Threshold Calibration Curve")
    ax.set_xlabel("Confidence Threshold")
    ax.set_ylabel("Rate")
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"Optimal Probability Threshold Selected: {best_thresh:.2f}")
    return best_thresh, calib_df

best_thresh, calib_df = calibrate_classifier_threshold(best_clf, scaler, encoder)


## 💾 Section 4: Model Persistence & Modular Inference Class

We serialize the trained classifier, scaler, encoder, and threshold manifest using `joblib` so that they can be loaded instantly for production inference.


In [ ]:
# 5. Save Artifacts & Define TrainedFaceClassifier
def save_model_artifacts(classifier, scaler, encoder, best_name, threshold):
    joblib.dump(classifier, MODELS_DIR / "best_classifier.joblib")
    joblib.dump(scaler, MODELS_DIR / "scaler.joblib")
    joblib.dump(encoder, MODELS_DIR / "label_encoder.joblib")
    with open(MODELS_DIR / "threshold.json", "w") as f:
        json.dump({"confidence_threshold": threshold, "classifier": best_name}, f, indent=2)
    print(f"Saved model artifacts to {MODELS_DIR}")

save_model_artifacts(best_clf, scaler, encoder, best_name, best_thresh)

class TrainedFaceClassifier:
    def __init__(self, models_dir=MODELS_DIR):
        self.classifier = joblib.load(models_dir / "best_classifier.joblib")
        self.scaler = joblib.load(models_dir / "scaler.joblib")
        self.encoder = joblib.load(models_dir / "label_encoder.joblib")
        with open(models_dir / "threshold.json") as f:
            meta = json.load(f)
        self.threshold = meta["confidence_threshold"]

    def predict_embedding(self, embedding: np.ndarray):
        scaled = self.scaler.transform([l2_normalize(embedding)])
        probas = self.classifier.predict_proba(scaled)[0]
        idx = int(np.argmax(probas))
        conf = float(probas[idx])
        identity = str(self.encoder.inverse_transform([idx])[0])
        accepted = conf >= self.threshold
        return identity if accepted else "NOT RECOGNIZED", conf, accepted

trained_pipeline = TrainedFaceClassifier()
print("Loaded TrainedFaceClassifier module successfully!")


## 🖼️ Section 5: End-to-End Visual Inference Demonstration

We run inference using the full pipeline:
`Image -> SCRFD Detection -> ArcFace Embedding -> StandardScaler -> Trained Classifier -> Calibrated Rejection Threshold -> Visual Annotation`


In [ ]:
# 6. Qualitative Visual Inference Demo
def draw_trainable_results(image_bgr: np.ndarray, results: list):
    annotated = image_bgr.copy()
    for r in results:
        x1, y1, x2, y2 = r["bbox"].astype(int)
        is_unknown = r["identity"] in {"UNKNOWN", "NOT RECOGNIZED"}
        color = (0, 0, 255) if is_unknown else (0, 255, 0)
        
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        
        label = f"{r['identity']} | Cls Conf: {r['confidence']:.2f} | Det: {r['det_score']:.2f}"
        font = cv2.FONT_HERSHEY_SIMPLEX
        (tw, th), bl = cv2.getTextSize(label, font, 0.45, 1)
        top = max(y1, th + 8)
        
        cv2.rectangle(annotated, (x1, top - th - bl - 4), (x1 + tw + 6, top), color, -1)
        cv2.putText(annotated, label, (x1 + 3, top - 3), font, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return annotated

sample_known = list(KNOWN_TEST_DIR.rglob("*.jpg"))[:2]
sample_unknown = list(UNKNOWN_TEST_DIR.rglob("*.jpg"))[:2]
test_files = sample_known + sample_unknown

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, p in enumerate(test_files):
    img = cv2.imread(str(p))
    faces = detector.detect(img)
    results = []
    for f in faces:
        if f["embedding"] is None: continue
        id_str, conf, accepted = trained_pipeline.predict_embedding(f["embedding"])
        results.append({"bbox": f["bbox"], "det_score": f["det_score"], "identity": id_str, "confidence": conf})
        
    ann = draw_trainable_results(img, results)
    ann_rgb = cv2.cvtColor(ann, cv2.COLOR_BGR2RGB)
    axes[idx].imshow(ann_rgb)
    axes[idx].set_title(f"Image: {p.name}\nGT: {p.parent.name}")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()


## ⚔️ Section 6: Comparative Benchmark Analysis (Baseline vs Trainable)

| Architectural Metric | Task 1: Pretrained Baseline (Cosine Similarity) | Task 2: Trainable ML Classifier (Linear/RBF SVM) |
| :--- | :--- | :--- |
| **Mathematical Basis** | Linear template centroid dot product ($\mathbf{g}_j \cdot \mathbf{x}$) | Supervised hyperplanes & probability estimation |
| **Training Required** | ❌ None (Pretrained features only) | ⚡ Fast scikit-learn training (~2 seconds) |
| **Class Separation** | Equidistant angular cosine metric | Optimized maximum-margin hyperplanes |
| **Open-Set Handling** | Static similarity threshold ($\tau = 0.35$) | Calibrated probability threshold ($\tau_{\text{cls}} = 0.70$) |
| **Scalability** | Linear scan $\mathcal{O}(K \cdot D)$ | Fast matrix dot product $\mathcal{O}(\text{Hyperplanes})$ |
| **Production Ready** | Good baseline / Cold-start | High confidence & robustness |


## 🏁 Section 7: Research Summary & Conclusions

1. **Supervised Classification Boost**:
   - Training a Linear/RBF SVM on top of ArcFace 512D embeddings significantly improves intra-class decision boundary fidelity compared to simple template averaging.
2. **Confidence Calibration**:
   - Probability threshold calibration effectively separates out-of-distribution unknown faces from enrolled gallery identities.
3. **Reproducibility & Serialization**:
   - All trained artifacts (`best_classifier.joblib`, `scaler.joblib`, `label_encoder.joblib`, `threshold.json`) are self-contained and ready for Google Colab and production deployment.
